[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/SkinTokens_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SkinTokens_Colab.ipynb)


# 🦴 SkinTokens — Mesh to Rig with TokenRig (VAST-AI, MIT)

Automated **skeleton generation + skinning weight prediction** for any 3D mesh, via a unified
autoregressive model over learned *SkinTokens*. This notebook packages the model from the
official Gradio demo at [🤗&nbsp;VAST-AI/SkinTokens](https://huggingface.co/spaces/VAST-AI/SkinTokens)
for **free Google Colab**, with a polished UI and a few extra ergonomic niceties (Drive cache,
batch mode, low-VRAM toggle, Colab-side file picker).

* **Paper:** [arXiv&nbsp;2602.04805](https://arxiv.org/abs/2602.04805) &nbsp;·&nbsp;
  **Code:** [VAST-AI-Research/SkinTokens](https://github.com/VAST-AI-Research/SkinTokens) &nbsp;·&nbsp;
  **Project page:** [zjp-shadow.github.io/works/SkinTokens](https://zjp-shadow.github.io/works/SkinTokens/)
* **Model card:** [🤗&nbsp;VAST-AI/SkinTokens](https://huggingface.co/VAST-AI/SkinTokens) (≈1.6 GB, MIT)
* SkinTokens is the **successor to [UniRig](https://github.com/VAST-AI-Research/UniRig) (SIGGRAPH '25)**.
  While UniRig decouples skeleton and skinning into two stages, SkinTokens unifies both into a
  **single autoregressive sequence** via learned discrete *skin tokens*, yielding
  **98%–133%** improvement in skinning accuracy and **17%–22%** improvement in bone prediction
  over state-of-the-art baselines.

## Method in three stages

1. **Learn SkinTokens** — an FSQ-CVAE compresses sparse skinning weights into a compact
   discrete vocabulary (codebook 64,000 entries).
2. **Unified autoregressive modeling** — a Qwen3-0.6B-based Transformer generates the
   full rig (skeleton followed by SkinTokens) as one interleaved sequence.
3. **RL refinement via GRPO** — tailored geometric and semantic rewards
   (volumetric joint coverage, bone-mesh containment, skinning sparsity, deformation smoothness)
   refine the model for out-of-distribution assets.

## What you get

* **Input:** one or more meshes in `.obj` / `.fbx` / `.glb`
* **Output:** a rigged `.glb` containing the predicted **skeleton + per-vertex skinning weights**
* **Two checkpoints** are downloaded (≈1.6 GB total): the GRPO-refined autoregressive model
  (`grpo_1400.ckpt`) and the FSQ-CVAE skin-weight tokenizer (`last.ckpt`)
* A small **Blender Python** (`bpy`) subprocess is booted once at startup to handle
  `.glb`/`.fbx`/`.obj` I/O. Its ~200&nbsp;MB shared object takes 30–60&nbsp;s to import, so we
  keep it in a long-lived sidecar (matches the official demo's `bpy_server` pattern)

## License

This notebook wraps the upstream **[VAST-AI-Research/SkinTokens](https://github.com/VAST-AI-Research/SkinTokens)**
codebase, which is released under the **MIT License**. The Blender `bpy` package
(used as a library) is **GPL-3.0**; redistribution here is via the upstream wheel
shipped in the public HF Space, with the same license and source-availability intact.
See the **[SkinTokens LICENSE](https://github.com/VAST-AI-Research/SkinTokens/blob/main/LICENSE)** for
the full text.

## Requirements

* **GPU:** NVIDIA, ≥ 14 GB VRAM recommended. The T4 on Colab free tier (15 GB) works
  with the **Low VRAM** mode enabled (CPU offload of the Qwen3 backbone).
* **RAM:** ≥ 25 GB recommended (Blender shared object, plus the FSQ-CVAE encoder & decoder)
* **Disk:** ≈ 8 GB free (PyTorch + CUDA, bpy wheel, model checkpoints, working space)
* **Time on first run:** 8–12 min (PyTorch + flash-attn + bpy wheel) + ≈ 3 min (checkpoint download)
* **Time on subsequent runs:** 2–3 min (everything cached in your Drive)

---

**Quick start:** run cells 2, 3, 4 in order, then either cell 7 (Gradio UI) or cell 8 (Colab-side batch).
The keep-alive cell 6 will keep the runtime alive for 12 hours.


In [ ]:
#@title STEP 1 — Install dependencies, clone repo, download checkpoints
"""
• Detects GPU + CUDA, installs torch 2.7.0 + cu128, flash-attn matching wheel
• Installs Blender as a Python module (bpy) — falls back to the HF-Space-bundled
  cp312 manylinux_2_39 wheel because PyPI only ships cp311 / cp313 bpy wheels
• Clones the official VAST-AI-Research/SkinTokens repo (MIT)
• Downloads the 2 checkpoints (≈1.6 GB) + Qwen3-0.6B config to Drive cache
• Boots a long-lived bpy_server subprocess on port 59876 for GLB/FBX/OBJ I/O
"""
import os, sys, time, json, subprocess, shutil, pathlib, traceback, glob, importlib
import requests

print('='*72)
print('SkinTokens / TokenRig — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — inference will fall back to CPU (very slow)')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/SkinTokens')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
else:
    drive_root = pathlib.Path('/content/_skintokens_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')

os.environ.setdefault('XFORMERS_IGNORE_FLASH_VERSION_CHECK', '1')
REPO_DIR  = drive_root / 'repo'
CKPT_DIR  = drive_root / 'experiments'
QWEN_DIR  = drive_root / 'models' / 'Qwen3-0.6B'
BPY_WHEEL = drive_root / 'bpy-4.5.4rc0-cp312-cp312-manylinux_2_39_x86_64.whl'
BPY_PORT  = 59876
BPY_SERVER = f'http://localhost:{BPY_PORT}'
t_total = time.time()

# 1. PyTorch + flash-attn (cu128, torch 2.7) ──────────────────────────
if torch is None or not torch.cuda.is_available() or not torch.__version__.startswith('2.7'):
    print('\n[1/6] Installing PyTorch 2.7.0 + cu128 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'torch==2.7.0', 'torchvision==0.22.0', 'torchaudio==2.7.0',
         '--index-url', 'https://download.pytorch.org/whl/cu128'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.0f}s')

print('\n[2/6] Installing flash-attn (cu12torch2.7 cxx11abiTRUE cp312) ...')
t0 = time.time()
FLASH_WHEEL = ('https://github.com/Dao-AILab/flash-attention/releases/download/'
               'v2.8.3/flash_attn-2.8.3+cu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', FLASH_WHEEL],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print('  flash-attn prebuilt failed — falling back to source build (slow)')
    print('  stderr tail:', r.stderr[-500:])
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', 'flash-attn', '--no-build-isolation'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  WARNING: flash-attn install failed — model will use eager attention (slower)')
print(f'  flash-attn done in {time.time()-t0:.0f}s')

# 2. bpy (Blender as a Python module) ──────────────────────────────
# PyPI only ships cp311 / cp313 bpy wheels; Colab is on cp312 so we fall back
# to the custom wheel from the public HF Space repo (matches the official demo).
def _ensure_bpy():
    try:
        import bpy  # noqa: F401
        print('  bpy already importable')
        return
    except Exception:
        pass
    print('\n[3/6] Installing bpy 4.5.4rc0 (Blender as Python module) ...')
    t0 = time.time()
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '--quiet', 'bpy==4.2.0'],
            stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
        )
        import bpy  # noqa: F401
        print(f'  bpy installed from PyPI in {time.time()-t0:.0f}s')
        return
    except Exception:
        pass
    if not BPY_WHEEL.exists() or BPY_WHEEL.stat().st_size < 1_000_000:
        from huggingface_hub import hf_hub_download
        bpy_wheel_path = hf_hub_download(
            repo_id='VAST-AI/SkinTokens', repo_type='space',
            filename='bpy-4.5.4rc0-cp312-cp312-manylinux_2_39_x86_64.whl',
            local_dir=str(drive_root),
        )
        shutil.move(bpy_wheel_path, str(BPY_WHEEL))
    is_real_zip = False
    try:
        with open(BPY_WHEEL, 'rb') as f:
            is_real_zip = f.read(4).startswith(b'PK')
    except Exception:
        pass
    if not is_real_zip:
        from huggingface_hub import hf_hub_download
        bpy_wheel_real = hf_hub_download(
            repo_id='VAST-AI/SkinTokens', repo_type='space',
            filename='bpy-4.5.4rc0-cp312-cp312-manylinux_2_39_x86_64.whl',
        )
        shutil.copy(bpy_wheel_real, str(BPY_WHEEL))
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', str(BPY_WHEEL)],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    import bpy  # noqa: F401
    print(f'  bpy wheel installed from HF Space in {time.time()-t0:.0f}s')

_ensure_bpy()

# 3. Other Python deps ──────────────────────────────────────
print('\n[4/6] Installing remaining Python dependencies ...')
t0 = time.time()
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'transformers==4.57.0', 'diffusers>=0.35.0', 'gradio==5.49.1',
     'python-box', 'einops', 'omegaconf', 'pytorch_lightning', 'lightning',
     'addict', 'timm', 'fast-simplification', 'trimesh', 'open3d', 'pyrender',
     'huggingface_hub', 'wandb', 'numpy>=1.26.0', 'bottle', 'tornado', 'cython'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f'  deps installed in {time.time()-t0:.0f}s')

# 4. Clone the SkinTokens repo (or reuse Drive cache) ──────────────
if not (REPO_DIR / '.git').exists():
    print(f'\n[5/6] Cloning VAST-AI-Research/SkinTokens into {REPO_DIR} ...')
    t0 = time.time()
    r = subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/VAST-AI-Research/SkinTokens.git',
         str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  git clone failed:', r.stderr[-500:])
        raise RuntimeError('git clone failed')
    print(f'  cloned in {time.time()-t0:.0f}s')
else:
    print(f'\n[5/6] Reusing cached repo at {REPO_DIR}')

sys.path.insert(0, str(REPO_DIR))

# 5. Download checkpoints + Qwen3-0.6B config ─────────────────
print('\n[6/6] Downloading SkinTokens checkpoints + Qwen3-0.6B config ...')
from huggingface_hub import hf_hub_download, snapshot_download
t0 = time.time()
needed = [
    'experiments/skin_vae_2_10_32768/last.ckpt',
    'experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt',
]
for rel in needed:
    target = CKPT_DIR / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 100_000_000:
        print(f'  cached : {rel}  ({target.stat().st_size // (1024*1024)} MB)')
        continue
    print(f'  download : {rel}  (1-3 min)')
    p = hf_hub_download(
        repo_id='VAST-AI/SkinTokens', filename=rel, local_dir=str(CKPT_DIR),
    )
    print(f'    -> {p}')
if not (QWEN_DIR / 'tokenizer.json').exists():
    print('  download : Qwen3-0.6B tokenizer + config (no weights)')
    snapshot_download(
        repo_id='Qwen/Qwen3-0.6B', local_dir=str(QWEN_DIR),
        ignore_patterns=['*.bin', '*.safetensors', 'onnx/*', '*.gguf'],
    )
else:
    print(f'  cached : {QWEN_DIR}')
print(f'  all checkpoints ready in {time.time()-t0:.0f}s')

# Symlinks so the loader's relative paths ('./experiments/...') resolve
exp_link = pathlib.Path('/content/experiments')
if not exp_link.exists():
    exp_link.symlink_to(CKPT_DIR / 'experiments')
qwen_link = pathlib.Path('/content/models')
if not qwen_link.exists():
    qwen_link.symlink_to(drive_root / 'models', target_is_directory=True)

# 6. Start the bpy_server subprocess ────────────────────────────
def _bpy_alive():
    try:
        return requests.get(f'{BPY_SERVER}/ping', timeout=1).status_code == 200
    except Exception:
        return False
if _bpy_alive():
    print('\n[boot] bpy_server already responding (cached from prior run)')
else:
    import atexit, signal
    bpy_log = open(drive_root / 'bpy_server.log', 'a')
    bpy_log.write(f'\n=== restart {time.time():.0f} ===\n')
    bpy_log.flush()
    proc = subprocess.Popen(
        [sys.executable, 'bpy_server.py'], cwd=str(REPO_DIR),
        stdout=bpy_log, stderr=bpy_log, preexec_fn=os.setsid,
    )
    print(f'\n[boot] bpy_server pid={proc.pid}, log={bpy_log.name}')
    def _cleanup():
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception:
            pass
    atexit.register(_cleanup)
    t0 = time.time()
    while time.time() - t0 < 180:
        if _bpy_alive():
            print(f'  bpy_server is ready (after {time.time()-t0:.0f}s)')
            break
        time.sleep(1.0)
    else:
        print(f'  WARNING: bpy_server did not respond within 180s; check {bpy_log.name}')

print()
print('='*72)
print(f'  STEP 1 complete  (total {time.time()-t_total:.0f}s)')
print('  Next: run STEP 2 (imports + lazy model load)')
print('='*72)


In [ ]:
#@title STEP 2 — Imports & lazy model load
"""
Loads the upstream `src.*` modules (one-time cost) and the cached model.
The bpy_server sidecar handles all `.glb`/`.fbx`/`.obj` I/O via HTTP.
Defines `run_rig(...)` which the UI and batch cells both call.
"""
import sys, os, time, json, pathlib, warnings, importlib
warnings.filterwarnings('ignore')

REPO_DIR = pathlib.Path(os.environ.get('AEI_SKIN_REPO',
                    str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/SkinTokens/repo'))))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR) if REPO_DIR.exists() else None
print(f'  repo        : {REPO_DIR}')
print(f'  experiments : {os.path.exists("experiments")}')
print(f'  models      : {os.path.exists("models")}')
print()

print('  Loading src.* modules (one-time cost) ...')
t0 = time.time()
try:
    from src.data.dataset import DatasetConfig, RigDatasetModule
    print(f'  src.data.dataset       ✓')
except Exception as e:
    print(f'  src.data.dataset       ✗  {type(e).__name__}: {e}')
    raise
from src.data.transform import Transform
print(f'  src.data.transform     ✓')
from src.model.tokenrig import TokenRig, TokenRigResult
print(f'  src.model.tokenrig     ✓')
from src.tokenizer.parse import get_tokenizer
print(f'  src.tokenizer.parse    ✓')
from src.server.spec import BPY_SERVER, BPY_PORT, get_model, object_to_bytes, bytes_to_object
print(f'  src.server.spec        ✓')
from src.data.vertex_group import voxel_skin
print(f'  src.data.vertex_group  ✓')
print(f'  total import: {time.time()-t0:.1f}s')
print()

import requests
try:
    r = requests.get(f'{BPY_SERVER}/ping', timeout=2)
    print(f'  bpy_server  ✓  port={BPY_PORT}  (responded: {r.status_code})')
except Exception as e:
    print(f'  bpy_server  ✗  {type(e).__name__}: {e}')
    print('  Re-run STEP 1 or `python bpy_server.py` manually in a cell.')
print()

import torch
from torch import Tensor
from tqdm import tqdm

# Lazy model cache — avoids re-loading on every Gradio click
_MODEL_CACHE = {}
def get_cached_model(model_ckpt, hf_path):
    key = (model_ckpt, hf_path)
    if key in _MODEL_CACHE:
        return _MODEL_CACHE[key]
    print(f'  Loading TokenRig from {model_ckpt} ...')
    t0 = time.time()
    model = get_model(model_ckpt, hf_path=hf_path)
    tokenizer = get_tokenizer(**model.tokenizer_config)
    transform = Transform.parse(**model.transform_config['predict_transform'])
    _MODEL_CACHE[key] = (model, tokenizer, transform)
    print(f'  model loaded in {time.time()-t0:.1f}s (cached for next call)')
    return _MODEL_CACHE[key]

def _ensure_bpy_alive():
    try:
        return requests.get(f'{BPY_SERVER}/ping', timeout=2).status_code == 200
    except Exception:
        return False

def run_rig(
    filepaths, top_k, top_p, temperature, repetition_penalty, num_beams,
    use_skeleton, use_transfer, use_postprocess, output_paths,
    model_ckpt, hf_path, low_vram=False, max_length=2048,
):
    if not _ensure_bpy_alive():
        raise RuntimeError(
            f'bpy_server is not responding on {BPY_SERVER}. Re-run STEP 1 to restart it.'
        )
    model, tokenizer, transform = get_cached_model(model_ckpt, hf_path or None)
    if low_vram:
        try:
            if hasattr(model, 'transformer') and hasattr(model.transformer, 'model'):
                model.transformer.model = model.transformer.model.cpu()
                print('  low_vram: moved Qwen3 backbone to CPU')
        except Exception as e:
            print(f'  low_vram offload skipped: {e}')

    datapath = {
        'data_name': None,
        'loader': 'bpy_server',
        'filepaths': {'articulation': [str(p) for p in filepaths]},
    }
    dataset_config = DatasetConfig.parse(
        shuffle=False, batch_size=1, num_workers=0,
        pin_memory=False, persistent_workers=False, datapath=datapath,
    ).split_by_cls()
    module = RigDatasetModule(
        predict_dataset_config=dataset_config,
        predict_transform=transform,
        tokenizer=tokenizer,
        process_fn=model._process_fn,
    )
    dataloader = module.predict_dataloader()['articulation']
    infer_device = model.device if model is not None else 'cuda'
    results_out = []
    for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
        batch = {k: v.to(infer_device) if isinstance(v, Tensor) else v
                 for k, v in batch.items()}
        if not use_skeleton:
            batch.pop('skeleton_tokens', None)
            batch.pop('skeleton_mask', None)
        batch['generate_kwargs'] = dict(
            max_length=int(max_length),
            top_k=int(top_k), top_p=float(top_p), temperature=float(temperature),
            repetition_penalty=float(repetition_penalty),
            num_return_sequences=1, num_beams=int(num_beams), do_sample=True,
        )
        if 'skeleton_tokens' in batch and 'skeleton_mask' in batch:
            mask = batch['skeleton_mask'][0] == 1
            skeleton_tokens = batch['skeleton_tokens'][0][mask].cpu().numpy()
        else:
            skeleton_tokens = None
        try:
            preds = model.predict_step(
                batch,
                skeleton_tokens=[skeleton_tokens] if skeleton_tokens is not None else None,
                make_asset=True,
            )['results']
        except Exception as e:
            print(f'  [{i:03d}] predict_step failed: {type(e).__name__}: {e}')
            traceback.print_exc()
            continue
        asset = preds[0].asset
        if asset is None:
            print(f'  [{i:03d}] no asset returned, skipping')
            continue
        if use_postprocess:
            try:
                voxel = asset.voxel(resolution=196)
                asset.skin *= voxel_skin(
                    grid=0, grid_coords=voxel.coords, joints=asset.joints,
                    vertices=asset.vertices, faces=asset.faces,
                    mode='square', voxel_size=voxel.voxel_size,
                )
                asset.normalize_skin()
            except Exception as e:
                print(f'  [{i:03d}] postprocess failed: {e}')
        out_path = output_paths[i]
        out_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            if use_transfer:
                payload = dict(
                    source_asset=asset,
                    target_path=asset.path,
                    export_path=str(out_path),
                    group_per_vertex=4,
                )
                res = bytes_to_object(requests.post(
                    f'{BPY_SERVER}/transfer', data=object_to_bytes(payload),
                ).content)
            else:
                payload = dict(asset=asset, filepath=str(out_path), group_per_vertex=4)
                res = bytes_to_object(requests.post(
                    f'{BPY_SERVER}/export', data=object_to_bytes(payload),
                ).content)
            if isinstance(res, dict) and 'error' in res:
                print(f'  [{i:03d}] bpy_server error: {res["error"]}')
                continue
            print(f'  [{i:03d}] OK  ->  {out_path}')
            results_out.append(out_path)
        except Exception as e:
            print(f'  [{i:03d}] export failed: {type(e).__name__}: {e}')
            traceback.print_exc()
    return results_out

print('  ready: run STEP 3 (form cell) to test, then STEP 4 for the Gradio UI')


In [ ]:
#@title STEP 3 — Core utilities (single-mesh helper, batch, Drive mirror)
"""
Re-exports run_rig from STEP 2 and adds:
  • `run_single(mesh_path, **kwargs)` — convenience for one mesh
  • `run_batch(input_dir, output_dir, **kwargs)` — process a whole folder
  • `_mirror_to_drive(paths, output_root)` — per-item Drive mirror with
    the same-path-skip that TripoSPlat / SplatTransform use.
"""
import shutil, traceback

def run_single(mesh_path, output_dir='/content/SkinTokens_Out', **kwargs):
    """Run the rig pipeline on a single mesh. Returns the output .glb path."""
    src = pathlib.Path(mesh_path).resolve()
    out_root = pathlib.Path(output_dir).resolve()
    out_root.mkdir(parents=True, exist_ok=True)
    out_path = out_root / (src.stem + '.glb')
    if out_path.exists() and out_path.stat().st_size > 1024:
        print(f'  already exists: {out_path}  ({out_path.stat().st_size//1024} KB)')
        return out_path
    results = run_rig(
        [src], kwargs.get('top_k', 5), kwargs.get('top_p', 0.95),
        kwargs.get('temperature', 1.0), kwargs.get('repetition_penalty', 2.0),
        kwargs.get('num_beams', 10), kwargs.get('use_skeleton', False),
        kwargs.get('use_transfer', False), kwargs.get('use_postprocess', False),
        [out_path], kwargs.get('model_ckpt', 'experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt'),
        kwargs.get('hf_path', 'models/Qwen3-0.6B'),
        low_vram=kwargs.get('low_vram', True),
        max_length=kwargs.get('max_length', 2048),
    )
    return results[0] if results else None

def run_batch(input_dir, output_dir, **kwargs):
    """Process every .obj/.fbx/.glb in `input_dir` (recursively) into `output_dir`.
    Preserves the directory layout; output extension is always .glb."""
    SUPP = {'.obj', '.fbx', '.glb', '.gltf'}
    in_root = pathlib.Path(input_dir).resolve()
    out_root = pathlib.Path(output_dir).resolve()
    out_root.mkdir(parents=True, exist_ok=True)
    files = sorted(p for p in in_root.rglob('*') if p.suffix.lower() in SUPP)
    if not files:
        print(f'  no supported files in {in_root}')
        return []
    outputs = []
    pending = []
    for f in files:
        rel = f.relative_to(in_root)
        out = (out_root / rel).with_suffix('.glb')
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.exists() and out.stat().st_size > 1024:
            outputs.append(out)
            continue
        pending.append((f, out))
    if not pending:
        print(f'  all {len(outputs)} already done')
        return outputs
    print(f'  {len(pending)} file(s) to process ({len(outputs)} skipped, already done)')
    f_list = [f for f, _ in pending]
    o_list = [o for _, o in pending]
    new = run_rig(
        f_list, kwargs.get('top_k', 5), kwargs.get('top_p', 0.95),
        kwargs.get('temperature', 1.0), kwargs.get('repetition_penalty', 2.0),
        kwargs.get('num_beams', 10), kwargs.get('use_skeleton', False),
        kwargs.get('use_transfer', False), kwargs.get('use_postprocess', False),
        o_list, kwargs.get('model_ckpt', 'experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt'),
        kwargs.get('hf_path', 'models/Qwen3-0.6B'),
        low_vram=kwargs.get('low_vram', True),
        max_length=kwargs.get('max_length', 2048),
    )
    outputs.extend(new)
    return outputs

def _mirror_to_drive(paths, output_root, drive_subdir='SkinTokens'):
    """Mirror each output file to /content/drive/MyDrive/AEI_3D_Out/<drive_subdir>/.
    Skips files whose source and destination resolve to the same path (e.g. when
    the user pointed the output_dir at a Drive path)."""
    if not paths:
        return
    drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out') / drive_subdir
    try:
        drive_base.parent.mkdir(parents=True, exist_ok=True)
        for src in paths:
            try:
                _src_path = pathlib.Path(src) if not isinstance(src, pathlib.Path) else src
                dst = drive_base / _src_path.relative_to(pathlib.Path(output_root))
                dst.parent.mkdir(parents=True, exist_ok=True)
                if _src_path.resolve() == dst.resolve():
                    continue
                if dst.exists() and dst.stat().st_size == _src_path.stat().st_size:
                    continue
                shutil.copy2(str(_src_path), str(dst))
            except Exception as e:
                print(f'  WARN: drive mirror of {_src_path.name} failed: {e}')
        print(f'  drive mirror: {drive_base}')
    except Exception as e:
        print(f'  drive mirror skipped: {e}')

print('  core helpers ready: run_single, run_batch, _mirror_to_drive')


In [ ]:
#@title STEP 4 — Launch Gradio UI (mirror of the official demo, with extras)
"""
Interactive Gradio UI matching the official demo:
  • Multi-file upload (.obj / .fbx / .glb)
  • Generation settings accordion: model path, Qwen3 path, sampling sliders,
    pipeline toggles (use_skeleton, use_transfer, use_postprocess),
    Colab toggles (low_vram, do_drive_save)
  • Run button → status textbox + downloadable .glb files
  • Output mirrored to /content/drive/MyDrive/AEI_3D_Out/SkinTokens/
  • Drive-mirror toggle in the accordion
Cell stays alive after launch (prevent_thread_lock=True).
"""
import os, sys, time, json, shutil, pathlib, traceback, tempfile
import requests
import gradio as gr

# Pre-warm the model so the first UI click is fast
try:
    _ = get_cached_model('experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt',
                          'models/Qwen3-0.6B')
    print('  model pre-warmed')
except Exception as e:
    print(f'  WARNING: model preload failed: {e}')
    print('  The UI will retry on first Run click.')

gr.TEMP_DIR = '/tmp/gradio_st'
os.makedirs(gr.TEMP_DIR, exist_ok=True)
_RUN_TOT = 0

def _do_run(files, top_k, top_p, temperature, rep_pen, num_beams,
            max_length, use_skel, use_xfer, use_pp, low_vram, do_drive_save):
    global _RUN_TOT
    if not files:
        return 'Please upload at least one 3D model.', None
    try:
        tmp_out = pathlib.Path(tempfile.mkdtemp(prefix='st_gradio_'))
        outputs = []
        filepaths = []
        for f in files:
            _RUN_TOT += 1
            src = pathlib.Path(f.name if hasattr(f, 'name') else f)
            filepaths.append(src)
            outputs.append(tmp_out / f'res_{_RUN_TOT}.glb')
        results = run_rig(
            filepaths, top_k, top_p, temperature, rep_pen, num_beams,
            use_skel, use_xfer, use_pp, outputs,
            'experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt',
            'models/Qwen3-0.6B',
            low_vram=low_vram, max_length=int(max_length),
        )
        if not results:
            return 'No results — see cell output above for errors.', None
        if do_drive_save:
            _mirror_to_drive(results, tmp_out)
        total_kb = sum(f.stat().st_size for f in results) // 1024
        return f'Processed {len(results)} mesh(es), total {total_kb} KB.', [str(r) for r in results]
    except Exception as e:
        traceback.print_exc()
        return f'Error: {type(e).__name__}: {e}', None

with gr.Blocks(title='SkinTokens · TokenRig (AEI Colab)') as demo:
    gr.Markdown(
        '''
        ## 🦴 Mesh to Rig with [SkinTokens](https://zjp-shadow.github.io/works/SkinTokens/) · TokenRig
        Automated **skeleton + skinning** for any 3D mesh. Successor to [UniRig](https://github.com/VAST-AI-Research/UniRig) (SIGGRAPH&nbsp;'25).
        * **Paper:** [arXiv&nbsp;2602.04805](https://arxiv.org/abs/2602.04805) &nbsp;·&nbsp;
          **Code:** [VAST-AI-Research/SkinTokens](https://github.com/VAST-AI-Research/SkinTokens) &nbsp;·&nbsp;
          **Weights:** [🤗&nbsp;VAST-AI/SkinTokens](https://huggingface.co/VAST-AI/SkinTokens)
        '''
    )
    gr.Markdown(
        '💡 *Defaults work well for most meshes. Enable **Use existing skeleton** if your mesh already has a rig. Enable **Preserve texture & scale** to keep your original textures.*'
    )
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(
                label='3D Models  ( .obj / .fbx / .glb )',
                file_count='multiple',
                file_types=['.obj', '.fbx', '.glb'],
            )
            with gr.Accordion('⚙️ Generation Settings', open=False):
                gr.Markdown('**Sampling parameters**')
                top_k = gr.Slider(1, 200, value=5, step=1,
                                  label='top_k',
                                  info='Sample from the K most likely next tokens. Lower = more deterministic.')
                top_p = gr.Slider(0.1, 1.0, value=0.95, step=0.01,
                                  label='top_p (nucleus)',
                                  info='Sample from the smallest token set with cumulative probability ≥ p.')
                temperature = gr.Slider(0.1, 2.0, value=1.0, step=0.1,
                                        label='temperature',
                                        info='<1 sharpens, >1 diversifies.')
                repetition_penalty = gr.Slider(0.5, 3.0, value=2.0, step=0.1,
                                               label='repetition_penalty',
                                               info='Penalty for tokens already generated. 1.0 = off.')
                num_beams = gr.Slider(1, 20, value=10, step=1,
                                      label='num_beams',
                                      info='Beam-search width. Larger = slower but higher quality. 1 disables beam search.')
                max_length = gr.Slider(256, 4096, value=2048, step=128,
                                       label='max_length',
                                       info='Maximum autoregressive sequence length. Larger = longer rigs at higher cost.')
                gr.Markdown('**Pipeline toggles**')
                use_skel = gr.Checkbox(False,
                                       label='Use existing skeleton (predict skin only)',
                                       info='Mesh already has a skeleton: keep it and only predict per-vertex skinning.')
                use_xfer = gr.Checkbox(False,
                                       label='Preserve original texture & scale',
                                       info='Transfer rig back onto the original (unprocessed) mesh — textures preserved.')
                use_pp = gr.Checkbox(False,
                                     label='Voxel skin post-processing',
                                     info='Apply a voxel-based mask to the predicted skin weights. Slower.')
                gr.Markdown('**Colab / T4 toggles**')
                low_vram = gr.Checkbox(True,
                                       label='Low VRAM (CPU offload Qwen3 — recommended for T4)',
                                       info='Move the Qwen3 backbone to CPU. Drops peak VRAM from ~14 GB to ~6-8 GB.')
                do_drive_save = gr.Checkbox(True,
                                            label='Mirror outputs to Google Drive',
                                            info='Copy each rigged .glb to /content/drive/MyDrive/AEI_3D_Out/SkinTokens/')
            run_btn = gr.Button('🚀 Run', variant='primary')
        with gr.Column(scale=1):
            log = gr.Textbox(label='Status', lines=2, interactive=False)
            output = gr.File(label='Rigged GLB output', interactive=False)
            gr.Markdown(
                '''
                **Notes**
                * The output `.glb` contains the predicted **skeleton + skinning weights**. Import it
                  in Blender (File → Import → glTF&nbsp;2.0) or any DCC tool that reads glTF.
                * In Blender, if you see a `glTF_not_exported` placeholder node, you can safely remove it.
                * On a T4 with Low VRAM on, each mesh takes ≈ 30–90 s. Disable Low VRAM only on A100/L4
                  for full GPU speed (peaks at ~14 GB VRAM).
                '''
            )
    run_btn.click(
        _do_run,
        inputs=[files, top_k, top_p, temperature, repetition_penalty, num_beams,
                max_length, use_skel, use_xfer, use_pp, low_vram, do_drive_save],
        outputs=[log, output],
    )
    def _welcome():
        return 'Ready. Upload one or more meshes and click Run.'
    demo.load(_welcome, outputs=[log])

from IPython.display import clear_output
clear_output()
demo.queue(default_concurrency_limit=2).launch(
    share=False, prevent_thread_lock=True, inline=True,
    show_error=True, height=900,
)
print('\n  Gradio UI running above ↑  (cell stays alive — do not re-run)')


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""
Keeps the runtime alive for 12 h so the Gradio UI stays reachable and the
bpy_server subprocess keeps its warm cache. Also prints a session summary
for quick verification.
"""
import datetime, time, json, sys, pathlib, os, requests
summary = {
    'timestamp'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'python'      : sys.version.split()[0],
    'torch'       : None, 'cuda': None, 'gpus': [],
    'bpy_server'  : None,
    'drive_cache' : str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/SkinTokens')),
}
try:
    import torch
    summary['torch'] = torch.__version__
    summary['cuda']  = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            summary['gpus'].append(f'{p.name}  {p.total_memory // (1024**3)} GB')
except Exception as e:
    summary['torch_error'] = str(e)
try:
    summary['bpy_server'] = requests.get(f'{BPY_SERVER}/ping', timeout=2).status_code == 200
except Exception:
    summary['bpy_server'] = False
print(json.dumps(summary, indent=2))
print()
print('  • STEP 4 is the Gradio UI (interactive multi-mesh)')
print('  • STEP 6 is the Colab-side single-mesh picker')
print('  • STEP 7 is the Colab-side batch (folder of meshes)')
print('  • Outputs land in OUTPUT_DIR and (if mirrored) in /content/drive/MyDrive/AEI_3D_Out/SkinTokens/')
print()
print('  Cell will keep the runtime alive for 12 h unless you disconnect.')
print('  Press the stop button in the toolbar to release the GPU.')

_start = time.time()
while time.time() - _start < 43200:  # 12 h
    time.sleep(300)
    print(f'  [{int(time.time()-_start)//60} min] still running — close tab to stop')


In [ ]:
#@title STEP 6 — Quick test (Colab-side file picker for a single mesh)
"""
Quick-test one mesh via the Colab file picker. Use this for verification
or for scripting (no Gradio UI). For interactive multi-mesh, use STEP 4.
"""
import time, pathlib, shutil

MESH_FILE     = ''  #@param {type:'string', placeholder: '/content/mesh.glb'}
OUTPUT_DIR    = '/content/SkinTokens_Out'  #@param {type:'string'}
MODEL_CKPT    = 'experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt'  #@param {type:'string'}
HF_PATH       = 'models/Qwen3-0.6B'  #@param {type:'string'}

TOP_K            = 5       #@param {type:'slider', min:1, max:200, step:1}
TOP_P            = 0.95    #@param {type:'slider', min:0.1, max:1.0, step:0.01}
TEMPERATURE      = 1.0     #@param {type:'slider', min:0.1, max:2.0, step:0.1}
REPETITION_PENALTY = 2.0   #@param {type:'slider', min:0.5, max:3.0, step:0.1}
NUM_BEAMS        = 10      #@param {type:'slider', min:1, max:20, step:1}
MAX_LENGTH       = 2048    #@param {type:'slider', min:256, max:4096, step:128}

USE_SKELETON     = False   #@param {type:'boolean'}
USE_TRANSFER     = False   #@param {type:'boolean'}
USE_POSTPROCESS  = False   #@param {type:'boolean'}
LOW_VRAM         = True    #@param {type:'boolean'}
DO_DRIVE_SAVE    = True    #@param {type:'boolean'}

if not MESH_FILE.strip():
    raise SystemExit('Set MESH_FILE to a .glb / .obj / .fbx path.')
src = pathlib.Path(MESH_FILE).resolve()
if not src.exists():
    raise SystemExit(f'File not found: {src}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / (src.stem + '.glb')
print(f'  input    : {src}  ({src.stat().st_size//1024} KB)')
print(f'  output   : {out_path}')
print(f'  ckpt     : {MODEL_CKPT}')
print(f'  low_vram : {LOW_VRAM}   do_drive_save: {DO_DRIVE_SAVE}')
print()
t0 = time.time()
result = run_single(str(src), output_dir=str(out_dir),
                    top_k=TOP_K, top_p=TOP_P, temperature=TEMPERATURE,
                    repetition_penalty=REPETITION_PENALTY, num_beams=NUM_BEAMS,
                    use_skeleton=USE_SKELETON, use_transfer=USE_TRANSFER,
                    use_postprocess=USE_POSTPROCESS, low_vram=LOW_VRAM,
                    max_length=MAX_LENGTH, model_ckpt=MODEL_CKPT, hf_path=HF_PATH)
elapsed = time.time() - t0
if result is None:
    print('  no result — check cell output for errors')
else:
    print(f'\n  done in {elapsed:.0f}s')
    print(f'  output: {result}  ({result.stat().st_size//1024} KB)')
    if DO_DRIVE_SAVE:
        _mirror_to_drive([result], str(out_dir))


In [ ]:
#@title STEP 7 — Batch process a folder of meshes
"""
Process every .obj / .fbx / .glb in `INPUT_DIR` (recursively) into `OUTPUT_DIR`.
Preserves the directory layout; output extension is always .glb.
Already-done outputs are skipped on re-run.
"""
import time, pathlib, shutil

INPUT_DIR     = ''  #@param {type:'string', placeholder: '/content/meshes/'}
OUTPUT_DIR    = '/content/SkinTokens_Out'  #@param {type:'string'}
MODEL_CKPT    = 'experiments/articulation_xl_quantization_256_token_4/grpo_1400.ckpt'  #@param {type:'string'}
HF_PATH       = 'models/Qwen3-0.6B'  #@param {type:'string'}

TOP_K            = 5       #@param {type:'slider', min:1, max:200, step:1}
TOP_P            = 0.95    #@param {type:'slider', min:0.1, max:1.0, step:0.01}
TEMPERATURE      = 1.0     #@param {type:'slider', min:0.1, max:2.0, step:0.1}
REPETITION_PENALTY = 2.0   #@param {type:'slider', min:0.5, max:3.0, step:0.1}
NUM_BEAMS        = 10      #@param {type:'slider', min:1, max:20, step:1}
MAX_LENGTH       = 2048    #@param {type:'slider', min:256, max:4096, step:128}

USE_SKELETON     = False   #@param {type:'boolean'}
USE_TRANSFER     = False   #@param {type:'boolean'}
USE_POSTPROCESS  = False   #@param {type:'boolean'}
LOW_VRAM         = True    #@param {type:'boolean'}
DO_DRIVE_SAVE    = True    #@param {type:'boolean'}

if not INPUT_DIR.strip():
    raise SystemExit('Set INPUT_DIR to a folder containing .obj / .fbx / .glb files.')
in_dir = pathlib.Path(INPUT_DIR).resolve()
if not in_dir.exists():
    raise SystemExit(f'Input folder not found: {in_dir}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
print(f'  input    : {in_dir}')
print(f'  output   : {out_dir}')
print(f'  ckpt     : {MODEL_CKPT}')
print(f'  low_vram : {LOW_VRAM}   do_drive_save: {DO_DRIVE_SAVE}')
print()
t0 = time.time()
results = run_batch(str(in_dir), str(out_dir),
                    top_k=TOP_K, top_p=TOP_P, temperature=TEMPERATURE,
                    repetition_penalty=REPETITION_PENALTY, num_beams=NUM_BEAMS,
                    use_skeleton=USE_SKELETON, use_transfer=USE_TRANSFER,
                    use_postprocess=USE_POSTPROCESS, low_vram=LOW_VRAM,
                    max_length=MAX_LENGTH, model_ckpt=MODEL_CKPT, hf_path=HF_PATH)
elapsed = time.time() - t0
if not results:
    print('  no results — check cell output for errors')
else:
    print(f'\n  done in {elapsed:.0f}s  ({elapsed/max(1,len(results)):.1f}s per mesh)')
    print(f'  {len(results)} file(s) in {out_dir}')
    if DO_DRIVE_SAVE:
        _mirror_to_drive(results, str(out_dir))
